Shared dataset: cleaning, labels, split, image cache
Owner: Ahmed

In [3]:
from google.colab import drive
drive.mount("/content/drive")
!git clone -q https://github.com/eyadstein/cxr_second_reader.git || git -C cxr_second_reader pull
%cd /content/cxr_second_reader
!pip install -q -r requirements.txt
import sys; sys.path.append("/content/cxr_second_reader")

Mounted at /content/drive
/content/cxr_second_reader
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import os, re, numpy as np, pandas as pd, kagglehub
from src.config import SHARED, INDIANA_SLUG
from src.evaluate import extract_labels
os.makedirs(SHARED, exist_ok=True)
kagglehub.login()
path = kagglehub.dataset_download(INDIANA_SLUG)
rep = pd.read_csv(f"{path}/indiana_reports.csv")
proj = pd.read_csv(f"{path}/indiana_projections.csv")
print(rep.columns.tolist()); rep.head()

Using Colab cache for faster access to the 'chest-xrays-indiana-university' dataset.
['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']


,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,"Pulmonary Disease, Chronic Obstructive;Bullous...","Pulmonary Disease, Chronic Obstructive;Bullous...","PA and lateral views of the chest XXXX, XXXX a...",XXXX-year-old XXXX with XXXX.,None available,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,Osteophyte/thoracic vertebrae/multiple/small;T...,Osteophyte;Thickening;Lung,Xray Chest PA and Lateral,Chest and nasal congestion.,NaN,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.


In [5]:
def clean(t):
    t = str(t) if pd.notna(t) else ""
    t = t.replace("XXXX", " ").lower()
    return re.sub(r"\s+", " ", t).strip()

for c in ["indication", "comparison", "findings", "impression"]:
    rep[c] = rep[c].map(clean)
rep = rep[(rep.findings != "") | (rep.impression != "")].drop_duplicates("uid")
rep["target_text"] = (rep.findings + ". " + rep.impression + ".").str.strip()
frontal = proj[proj.projection == "Frontal"].drop_duplicates("uid")[["uid", "filename"]]
df = rep.merge(frontal, on="uid").rename(columns={"filename": "frontal_img"})

In [6]:
rng = np.random.default_rng(42)
uids = df.uid.unique(); rng.shuffle(uids); n = len(uids)
split = {u: "train" for u in uids[:int(.7 * n)]}
split.update({u: "val" for u in uids[int(.7 * n):int(.85 * n)]})
split.update({u: "test" for u in uids[int(.85 * n):]})
df["split"] = df.uid.map(split)
print(df.split.value_counts())

split
train    2566
test      550
val       550
Name: count, dtype: int64


In [7]:
# TODO labels_8: df["labels_8"] = (df.findings + " " + df.impression).map(lambda t: extract_labels(t).tolist())
# TODO image cache: save each frontal image as grayscale PNG into SHARED/img224 and SHARED/img384, named uid.png, with CLAHE. No horizontal flips anywhere.
df.to_csv(f"{SHARED}/dataset.csv", index=False)

In [ ]:
import json
imgdir = os.path.dirname(next(iter(files.values())))
os.makedirs(f"{SHARED}/llava", exist_ok=True)
Q = "<image>\nProvide a description of the findings in the radiology image."
for s in ["train", "val", "test"]:
    d = df[(df.split == s) & (df.findings != "")]
    rows = [{"id": str(r.uid), "image": r.frontal_img,
             "conversations": [{"from": "human", "value": Q}, {"from": "gpt", "value": r.findings}]}
            for r in d.itertuples()]
    json.dump(rows, open(f"{SHARED}/llava/{s}.json", "w"))
    print(s, len(rows))
print("image folder:", imgdir)